<a href="https://colab.research.google.com/github/Ratludu/Backpack-Prediction-Challenge/blob/main/Backpack_Prices_LB_38.84892.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [2]:
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('kaggleusername')
os.environ['KAGGLE_KEY'] = userdata.get('kaggleapi')

competition = 'playground-series-s5e2'

!kaggle competitions download -c {competition}

!unzip "{competition}.zip"

 98% 91.0M/92.7M [00:05<00:00, 24.4MB/s]
100% 92.7M/92.7M [00:05<00:00, 18.8MB/s]
Archive:  playground-series-s5e2.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               
  inflating: training_extra.csv      


In [3]:
!pip install dask-cuda==24.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.4/134.4 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.5/244.5 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2024.10.0
    Uninstalling dask-2024.10.0:
      Successfully uninstalled dask-2024.10.0


In [4]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 577, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 577 (delta 116), reused 82 (delta 82), pack-reused 434 (from 3)
Receiving objects: 100% (577/577), 188.95 KiB | 18.89 MiB/s, done.
Resolving deltas: 100% (290/290), done.
Installing RAPIDS remaining 24.12.* libraries
Using Python 3.11.11 environment at: /usr
Resolved 154 packages in 9.85s
 Downloaded ucx-py-cu12
 Downloaded libucx-cu12
 Downloaded datashader
 Downloaded cuspatial-cu12
 Downloaded cucim-cu12
 Downloaded scikit-image
 Downloaded libcuspatial-cu12
 Downloaded raft-dask-cu12
 Downloaded cuml-cu12
 Downloaded cuvs-cu12
 Downloaded cugraph-cu12
Prepared 21 packages in 17.64s
Uninstalled 1 package in 22ms
Installed 21 packages in 18ms
 + cucim-cu12==24.12.0
 + cugraph-cu12==24.12.0
 + cuml-cu12==24.12.0
 + cuproj-cu12==24.12.0
 + cuspatial-cu12==24.12.0
 + cuvs-cu12==24.12.0
 + cuxfilter-cu1

In [5]:
!pip install catboost
!pip install optuna
!pip install scikit-learn
!pip install numpy
!pip install seaborn
!pip install matplotlib
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.4/383.4 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 8.7 MB/s eta 0:00:00


In [6]:
import pandas as pd
import numpy as np
from numpy import random
from cuml.preprocessing import TargetEncoder
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [7]:
class config:
    # data links
    train_link = "train.csv"
    train_ex_link = "training_extra.csv"
    test_link = "test.csv"
    sub_link = "sample_submission.csv"

    # create folds config

    n_splits = 25

    # Ignore Columns

    col_ignore = ["id", "Price"]
    num_cols = ["Weight Capacity (kg)"]
    # target

    submit = True

    target = "Price"

    add_original = False

In [8]:
def rmse(y_true, y_pred):
    error = 0

    for yt, yp in zip(y_true, y_pred):
        error += (yt - yp) ** 2

    m = np.sqrt(error / len(y_true))

    return m

In [9]:
def random_columns(columns):

  # Generate random number for how many columns we want to concat
  rand_num = np.random.randint(2,5)

  # choose the columns from the list of columns with no repeats
  rand_cols = []
  for i in range(rand_num):
    col = np.random.choice(columns)
    while col in rand_cols:
      col = np.random.choice(columns)
    rand_cols.append(col)

  # return a list of the columns

  return "-".join(col for col in rand_cols),rand_cols


In [10]:
train = pd.read_csv(config.train_link)
train_ex = pd.read_csv(config.train_ex_link)
test = pd.read_csv(config.test_link)

In [11]:
train = pd.concat([train,train_ex], axis = 0, ignore_index = True)

In [15]:
kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)

drop = ['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment', 'Color', 'Waterproof']
added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
oof = np.zeros(len(train))
preds = np.zeros(len(test))
features = [col for col in train.columns if col not in config.col_ignore]
cats = [col for col in features if col not in config.num_cols]
cats.extend(added_fe)
m = []
for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

    x_train, x_val = train.loc[train_idx, features].copy(), train.loc[test_idx, features].copy()
    y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
    x_test = test[features].copy()

    TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


    # adding extra features

    x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
    x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
    x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

    x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
    x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
    x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')

    x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
    x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
    x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')

    for col in added_fe:
      TE.fit(x_train[col], y_train)
      x_train[f'{col}_TE'] = TE.transform(x_train[col])
      x_val[f'{col}_TE'] = TE.transform(x_val[col])
      x_test[f'{col}_TE'] = TE.transform(x_test[col])

    for col in features:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_TE'] = TE.transform(x_train[col])
        x_val[f'{col}_TE'] = TE.transform(x_val[col])
        x_test[f'{col}_TE'] = TE.transform(x_test[col])

    x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
    x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
    x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

    x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

    for cat in cats:
        x_train[cat] =  x_train[cat].fillna("MISSING")
        x_val[cat] = x_val[cat].fillna("MISSING")
        x_test[cat] = x_test[cat].fillna("MISSING")
        x_train[cat] =  x_train[cat].astype('str')
        x_val[cat] = x_val[cat].astype('str')
        x_test[cat] = x_test[cat].astype('str')

    print(x_train.columns)

    model = CatBoostRegressor(learning_rate = 0.11509572776170199,
                              task_type = "GPU",
                              grow_policy = 'Lossguide',
                              random_state = 42,
                              cat_features = cats,
                              verbose = 250,
                              loss_function='RMSE')

    model.fit(x_train, y_train)

    val_preds = model.predict(x_val)

    oof[test_idx] = val_preds

    preds += model.predict(x_test)/config.n_splits

    score = rmse(y_val, val_preds)

    m.append(score)

    print(f'Fold: {fold+1}, Score: {score}')

print(f"The average CV is {np.average(m)}")

Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'size-laptop compartment_fe_TE',
       'Color-waterproof_fe_TE', 'weightcapacity-color_fe_TE', 'Brand_TE',
       'Material_TE', 'Size_TE', 'Compartments_TE', 'Laptop Compartment_TE',
       'Waterproof_TE', 'Style_TE', 'Color_TE', 'Weight Capacity (kg)_TE',
       'colorxweight', 'Sizexweight', 'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8870251	total: 19.5ms	remaining: 19.5s
250:	learn: 38.6227726	total: 3.53s	remaining: 10.5s
500:	learn: 38.5941250	total: 6.75s	remaining: 6.72s
750:	learn: 38.5685236	total: 9.95s	remaining: 3.3s
999:	learn: 38.5457718	total: 13.2s	remaining: 0us
Fold: 1, Score: 38.639697293648744
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weigh

In [ ]:
save_columns ={}
save_score = {}

added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
for trial in range(10):

  features = [col for col in train.columns if col not in config.col_ignore]
  cats = [col for col in features if col not in config.num_cols]
  cats.extend(added_fe)

  randf,randc = random_columns(features)

  print(f"Trial {trial +1}: {randc}")

  kf = KFold(n_splits = 5, shuffle = True, random_state = 42)

  oof = np.zeros(len(train))
  preds = np.zeros(len(test))
  cats.append(randf)
  m = []
  for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

      x_train, x_val = train.loc[train_idx, features].copy(), train.loc[test_idx, features].copy()
      y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
      x_test = test[features].copy()

      TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


      # adding extra features

      x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
      x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
      x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

      x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
      x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
      x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')


      x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
      x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
      x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')

      for col in added_fe:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_TE'] = TE.transform(x_train[col])
        x_val[f'{col}_TE'] = TE.transform(x_val[col])
        x_test[f'{col}_TE'] = TE.transform(x_test[col])

      # random features

      x_train[randf] = "FE_"
      x_val[randf] = "FE_"
      x_test[randf] = "FE_"

      for col in randc:
          x_train[randf] += x_train[col].astype('str')
          x_val[randf] += x_val[col].astype('str')
          x_test[randf] += x_test[col].astype('str')

      TE.fit(x_train[randf], y_train)
      x_train[f'{randf}_TE'] = TE.transform(x_train[randf])
      x_val[f'{randf}_TE'] = TE.transform(x_val[randf])
      x_test[f'{randf}_TE'] = TE.transform(x_test[randf])

      for col in features:
          TE.fit(x_train[col], y_train)
          x_train[f'{col}_TE'] = TE.transform(x_train[col])
          x_val[f'{col}_TE'] = TE.transform(x_val[col])
          x_test[f'{col}_TE'] = TE.transform(x_test[col])


      x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
      x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
      x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

      x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
      x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
      x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

      x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
      x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
      x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

      x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
      x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
      x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

      for cat in cats:
          x_train[cat] =  x_train[cat].fillna("MISSING")
          x_val[cat] = x_val[cat].fillna("MISSING")
          x_test[cat] = x_test[cat].fillna("MISSING")
          x_train[cat] =  x_train[cat].astype('str')
          x_val[cat] = x_val[cat].astype('str')
          x_test[cat] = x_test[cat].astype('str')

      model = CatBoostRegressor(task_type = "GPU",
                                grow_policy = 'Lossguide',
                                random_state = 42,
                                cat_features = cats,
                                verbose = 250,
                                loss_function='RMSE')

      model.fit(x_train, y_train)

      val_preds = model.predict(x_val)

      oof[test_idx] = val_preds

      preds += model.predict(x_test)/config.n_splits

      score = rmse(y_val, val_preds)

      m.append(score)

      print(f'Fold: {fold+1}, Score: {score}')

  print(f"The average CV is {np.average(m)}, with {randf}")
  cats.pop()

  save_columns[randf] = randc
  save_score[randf] = np.average(m)

In [ ]:

def objective(trial):

    params = {
        "learning_rate": 0.11509572776170199,
        #"subsample": trial.suggest_float("subsample", 0.05, 1.0),
        #"colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.05, 1.0),
        #"min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
    }
    kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)

    drop = ['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment', 'Color', 'Waterproof']
    added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
    oof = np.zeros(len(train))
    preds = np.zeros(len(test))
    features = [col for col in train.columns if col not in config.col_ignore]
    cats = [col for col in features if col not in config.num_cols]
    cats.extend(added_fe)
    m = []
    for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

        x_train, x_val = train.loc[train_idx, features].copy(), train.loc[test_idx, features].copy()
        y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
        x_test = test[features].copy()

        TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


        # adding extra features

        x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
        x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
        x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

        x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
        x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
        x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')

        x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
        x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
        x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')

        for col in added_fe:
          TE.fit(x_train[col], y_train)
          x_train[f'{col}_TE'] = TE.transform(x_train[col])
          x_val[f'{col}_TE'] = TE.transform(x_val[col])
          x_test[f'{col}_TE'] = TE.transform(x_test[col])

        for col in features:
            TE.fit(x_train[col], y_train)
            x_train[f'{col}_TE'] = TE.transform(x_train[col])
            x_val[f'{col}_TE'] = TE.transform(x_val[col])
            x_test[f'{col}_TE'] = TE.transform(x_test[col])

        x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
        x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
        x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

        x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
        x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
        x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

        x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
        x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
        x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

        x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
        x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
        x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

        for cat in cats:
            x_train[cat] =  x_train[cat].fillna("MISSING")
            x_val[cat] = x_val[cat].fillna("MISSING")
            x_test[cat] = x_test[cat].fillna("MISSING")
            x_train[cat] =  x_train[cat].astype('str')
            x_val[cat] = x_val[cat].astype('str')
            x_test[cat] = x_test[cat].astype('str')

        print(x_train.columns)

        model = CatBoostRegressor(**params,
                                  task_type = "GPU",
                                  grow_policy = 'Lossguide',
                                  random_state = 42,
                                  cat_features = cats,
                                  verbose = 250,
                                  loss_function='RMSE')

        model.fit(x_train, y_train)

        val_preds = model.predict(x_val)

        oof[test_idx] = val_preds

        preds += model.predict(x_test)/config.n_splits

        score = rmse(y_val, val_preds)

        m.append(score)

        print(f'Fold: {fold+1}, Score: {score}')

    print(f"The average CV is {np.average(m)}")
    return np.average(m)

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5)

[I 2025-02-11 22:30:36,263] A new study created in memory with name: no-name-9fcf282b-a6fd-4192-887b-90902af589a9


Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'size-laptop compartment_fe_TE',
       'Color-waterproof_fe_TE', 'weightcapacity-color_fe_TE', 'Brand_TE',
       'Material_TE', 'Size_TE', 'Compartments_TE', 'Laptop Compartment_TE',
       'Waterproof_TE', 'Style_TE', 'Color_TE', 'Weight Capacity (kg)_TE',
       'colorxweight', 'Sizexweight', 'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8875230	total: 23.1ms	remaining: 23.1s
250:	learn: 38.6291798	total: 3.42s	remaining: 10.2s
500:	learn: 38.6044715	total: 6.8s	remaining: 6.78s
750:	learn: 38.5833778	total: 10.3s	remaining: 3.4s
999:	learn: 38.5639276	total: 13.6s	remaining: 0us
Fold: 1, Score: 38.64011593864008
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight 

[I 2025-02-11 22:55:14,940] Trial 0 finished with value: 38.6550181008312 and parameters: {'depth': 5}. Best is trial 0 with value: 38.6550181008312.


Fold: 25, Score: 38.746116489706075
The average CV is 38.6550181008312
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'size-laptop compartment_fe_TE',
       'Color-waterproof_fe_TE', 'weightcapacity-color_fe_TE', 'Brand_TE',
       'Material_TE', 'Size_TE', 'Compartments_TE', 'Laptop Compartment_TE',
       'Waterproof_TE', 'Style_TE', 'Color_TE', 'Weight Capacity (kg)_TE',
       'colorxweight', 'Sizexweight', 'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8875230	total: 17.7ms	remaining: 17.7s
250:	learn: 38.6286907	total: 3.4s	remaining: 10.2s
500:	learn: 38.6039786	total: 6.74s	remaining: 6.71s
750:	learn: 38.5827652	total: 10.1s	remaining: 3.33s
999:	learn: 38.5639206	total: 13.4s	remaining: 0us
Fold: 1, Score: 38.640022279935565
Index(['Brand', 'Material', 'Size', 'Compartment

[I 2025-02-11 23:19:50,957] Trial 1 finished with value: 38.65510967419889 and parameters: {'depth': 5}. Best is trial 0 with value: 38.6550181008312.


Fold: 25, Score: 38.748311944192125
The average CV is 38.65510967419889
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'size-laptop compartment_fe_TE',
       'Color-waterproof_fe_TE', 'weightcapacity-color_fe_TE', 'Brand_TE',
       'Material_TE', 'Size_TE', 'Compartments_TE', 'Laptop Compartment_TE',
       'Waterproof_TE', 'Style_TE', 'Color_TE', 'Weight Capacity (kg)_TE',
       'colorxweight', 'Sizexweight', 'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8894732	total: 13.5ms	remaining: 13.5s
250:	learn: 38.6462478	total: 2.56s	remaining: 7.63s
500:	learn: 38.6321316	total: 5.06s	remaining: 5.04s
750:	learn: 38.6210993	total: 7.59s	remaining: 2.52s
999:	learn: 38.6118915	total: 10.2s	remaining: 0us
Fold: 1, Score: 38.64012858176662
Index(['Brand', 'Material', 'Size', 'Compartmen

[I 2025-02-11 23:43:01,176] Trial 2 finished with value: 38.65476170071741 and parameters: {'depth': 4}. Best is trial 2 with value: 38.65476170071741.


Fold: 25, Score: 38.744620396106995
The average CV is 38.65476170071741
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'size-laptop compartment_fe_TE',
       'Color-waterproof_fe_TE', 'weightcapacity-color_fe_TE', 'Brand_TE',
       'Material_TE', 'Size_TE', 'Compartments_TE', 'Laptop Compartment_TE',
       'Waterproof_TE', 'Style_TE', 'Color_TE', 'Weight Capacity (kg)_TE',
       'colorxweight', 'Sizexweight', 'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8894732	total: 14.1ms	remaining: 14.1s
250:	learn: 38.6462478	total: 2.63s	remaining: 7.86s
500:	learn: 38.6320866	total: 5.17s	remaining: 5.15s
750:	learn: 38.6209368	total: 7.78s	remaining: 2.58s
999:	learn: 38.6115215	total: 10.4s	remaining: 0us
Fold: 1, Score: 38.64082853011884
Index(['Brand', 'Material', 'Size', 'Compartmen

[I 2025-02-12 00:06:13,554] Trial 3 finished with value: 38.65503738025785 and parameters: {'depth': 4}. Best is trial 2 with value: 38.65476170071741.


Fold: 25, Score: 38.7459920590337
The average CV is 38.65503738025785
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'size-laptop compartment_fe_TE',
       'Color-waterproof_fe_TE', 'weightcapacity-color_fe_TE', 'Brand_TE',
       'Material_TE', 'Size_TE', 'Compartments_TE', 'Laptop Compartment_TE',
       'Waterproof_TE', 'Style_TE', 'Color_TE', 'Weight Capacity (kg)_TE',
       'colorxweight', 'Sizexweight', 'Sizexbrand', 'Materialxweight'],
      dtype='object')
0:	learn: 38.8875230	total: 18.3ms	remaining: 18.3s
250:	learn: 38.6291228	total: 3.38s	remaining: 10.1s
500:	learn: 38.6041568	total: 6.74s	remaining: 6.71s
750:	learn: 38.5832636	total: 10.1s	remaining: 3.34s
999:	learn: 38.5645144	total: 13.4s	remaining: 0us
Fold: 1, Score: 38.64044755230158
Index(['Brand', 'Material', 'Size', 'Compartments

In [ ]:
print('Best hyperparameters:', study.best_params)
print('Best RMSE:', study.best_value)

NameError: name 'study' is not defined

In [ ]:
from google.colab import runtime
runtime.unassign()

In [16]:
submission = pd.read_csv(config.sub_link)
submission[config.target] = preds
submission.to_csv("submission.csv", index = False)

submission

,id,Price
0,300000,81.392468
1,300001,82.884870
2,300002,88.565724
3,300003,78.878812
4,300004,79.344333
...,...,...
199995,499995,81.861934
199996,499996,72.498241
199997,499997,83.021313
199998,499998,82.183635


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=model.get_feature_importance(), y=x_train.columns)
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()

In [17]:
if config.submit:
  !kaggle competitions submit -c {competition} -f submission.csv -m 'Submission with 20 kfold TE combo'

100% 4.74M/4.74M [00:02<00:00, 2.03MB/s]
Successfully submitted to Backpack Prediction Challenge

In [ ]:
!kaggle competitions submissions -c {competition}